In [20]:
!pip install -U pageindex openai python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Tarunvarma\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [21]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:   ", "✅" if OPENAI_API_KEY else "❌ Missing!")

PageIndex key loaded: ✅
OpenAI key loaded:    ✅


In [ ]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key="ur_api_key")
groq_client = Groq(
    api_key="ur_api_key")

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


In [23]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./sekiro.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./sekiro.pdf
✅ Uploaded!
📋 Document ID: pi-cmq9d27c8038701pcbk7iap0a
   (Save this ID — you'll use it throughout the notebook)


In [24]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break

    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: completed

✅ Tree index ready!


### Section 3: Inspect the Tree Structure

In [25]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 9

🌲 Raw tree (first node):
{
  "title": "Overview",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# Overview\n\nSekiro: Shadows Die Twice is an action-adventure game developed by FromSoftware and published by Activision, released on March 22, 2019. Set in a reimagined late Sengoku-era Japan, it follows a shinobi named Wolf (Ookami) on a mission to rescue his kidnapped lord and take revenge on his enemies. Unlike FromSoftware's Souls series, Sekiro emphasizes precise sword combat, stealth, and vertical traversal using a grappling hook prosthetic arm. It won the prestigious Game of the Year 2019 at The Game Awards.\n",
  "text": "# Overview\n\nSekiro: Shadows Die Twice is an action-adventure game developed by FromSoftware and published by Activision, released on March 22, 2019. Set in a reimagined late Sengoku-era Japan, it follows a shinobi named Wolf (Ookami) on a mission to rescue his kidnapped lord and take revenge on his enemies. Unlike FromSoftware's 

In [26]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Overview  (p.1)
[0001] Quick Facts  (p.1)
[0002] Story &amp; Setting  (p.1)
[0003] Gameplay Mechanics  (p.1)
[0004] Key Characters  (p.2)
[0005] Most Iconic Bosses  (p.2)
[0006] Multiple Endings  (p.2)
[0007] Awards &amp; Legacy  (p.2)
[0008] Did You Know?  (p.2)


In [27]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total


total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 9
   Each node = one retrievable section of the document


### Section 4: LLM Tree Search — The Core of PageIndex

In [28]:
def llm_tree_search(query: str, tree: list, model: str = "qwen/qwen3-32b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """

    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [29]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "Who is the villain of the sekiro and name them"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: Who is the villain of the sekiro and name them



🧠 LLM Reasoning:
The query asks for the villain of Sekiro. To find the best section to look in:
1. The most likely sections would involve characters (including antagonists) or major bosses.
2. Node 0005 ('Most Iconic Bosses') explicitly lists Genichiro Ashina as a boss (the game's primary antagonist).
3. Node 0004 ('Key Characters') provides character summaries but doesn't mention villains in the visible summary.
4. Node 0002 ('Story & Setting') provides background but not direct villain information.
Only node 0005 directly references the main villain in its summary.

🎯 Selected Node IDs: ['0005']


### Section 5: Full End-to-End RAG Pipeline

In [30]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [35]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "qwen/qwen3-32b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    groq_client.chat.completions.create
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [36]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

In [37]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="what's so special about this game?",
    tree=pageindex_tree
)

🔍 Query: what's so special about this game?

🧠 Reasoning: The query asks about what makes the game special. The most relevant sections are the 'Gameplay Mechanics' (0003) which introduces the unique Posture system, and 'Most Iconic Bosses' (0005) which highl...
🎯 Retrieved node IDs: ['0003', '0005']
📄 Sections found: ['Gameplay Mechanics', 'Most Iconic Bosses']

📝 Answer:
<think>
Okay, so the user is asking what's special about this game. Let me check the provided context to form an answer.

First, the context has two sections: Gameplay Mechanics and Most Iconic Bosses. The answer should focus on what makes the game unique. 

Looking at "Gameplay Mechanics" on Page 1, the Posture System is highlighted where breaking an enemy's balance is key instead of just HP. That's a standout feature. Also, the Deflection and Parry system is core to combat, which requires precise timing. The Shinobi Prosthetic with interchangeable tools adds versatility. The Resurrection ability allows for comebacks